<a href="https://colab.research.google.com/github/zegavibiano22-tech/WiFi-CSI-CrossDomain-Sensing/blob/main/notebooks/%2003_proposed_dann.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# 1. Clone repositori & masuk ke folder proyek
!git clone https://github.com/zegavibiano22-tech/WiFi-CSI-CrossDomain-Sensing.git
%cd WiFi-CSI-CrossDomain-Sensing

# 2. Import library utama & modul dataset
import os
import sys
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from src.dataset import WidarBVPDataset

# 3. Cek ketersediaan GPU/CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

Cloning into 'WiFi-CSI-CrossDomain-Sensing'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 62 (delta 27), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 52.92 KiB | 1.15 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/WiFi-CSI-CrossDomain-Sensing/WiFi-CSI-CrossDomain-Sensing/WiFi-CSI-CrossDomain-Sensing/WiFi-CSI-CrossDomain-Sensing
Menggunakan device: cpu


Kode itu berfungsi mengunduh repositori dari GitHub ke lingkungan Colab, masuk ke dalam folder proyeknya, memuat library PyTorch dan modul WidarBVPDataset, serta mengecek apakah program akan dijalankan memakai GPU (cuda) atau CPU.

In [13]:
from torch.autograd import Function

# 1. Gradient Reversal Layer (GRL)
class ReverseLayerF(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

# 2. Arsitektur Proposed DANN
class ProposedDANN(nn.Module):
    def __init__(self, num_classes=6, num_domains=3):
        super(ProposedDANN, self).__init__()

        # Feature Extractor (2D-CNN)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Output: (32, 10, 10)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Output: (64, 5, 5)
        )

        self.fc_bridge = nn.Linear(64 * 5 * 5, 128)
        self.lstm = nn.LSTM(input_size=128, hidden_size=64, num_layers=1, batch_first=True)

        # Label Predictor (Gesture Classifier)
        self.gesture_classifier = nn.Linear(64, num_classes)

        # Domain Classifier
        self.domain_classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_domains)
        )

    def forward(self, x, alpha=1.0):
        batch_size, channels, time_steps, height, width = x.size()

        # Process CNN
        x = x.permute(0, 2, 1, 3, 4).contiguous()
        x = x.view(batch_size * time_steps, channels, height, width)

        cnn_out = self.cnn(x)
        cnn_out = cnn_out.view(cnn_out.size(0), -1)
        features = self.fc_bridge(cnn_out)

        # Process LSTM
        features = features.view(batch_size, time_steps, -1)
        lstm_out, _ = self.lstm(features)
        feature_vector = lstm_out[:, -1, :] # Feature representation

        # Branch 1: Gesture Prediction
        out_gesture = self.gesture_classifier(feature_vector)

        # Branch 2: Domain Prediction (dengan GRL)
        reverse_features = ReverseLayerF.apply(feature_vector, alpha)
        out_domain = self.domain_classifier(reverse_features)

        return out_gesture, out_domain

# Tes Forward Pass
dann_model = ProposedDANN(num_classes=6, num_domains=3).to(device)
dummy_x = torch.randn(2, 1, 20, 20, 20).to(device)
out_g, out_d = dann_model(dummy_x, alpha=0.1)

print("Arsitektur Proposed DANN Berhasil Dibuat!")
print("Shape Output Gesture:", out_g.shape) # Expect: [2, 6]
print("Shape Output Domain:", out_d.shape)  # Expect: [2, 3]

Arsitektur Proposed DANN Berhasil Dibuat!
Shape Output Gesture: torch.Size([2, 6])
Shape Output Domain: torch.Size([2, 3])


**Program tersebut berfungsi membuat dan menguji arsitektur model Domain Adversarial Neural Network (DANN).**

Fungsinya:

Gradient Reversal Layer (GRL): Membalikkan arah gradien saat proses backpropagation agar fitur yang diekstrak model menjadi domain-invariant (tidak terikat pada lingkungan tertentu).

Feature Extractor (CNN + LSTM): Menggabungkan 2D-CNN untuk mengekstrak fitur spasial dari tiap frame BVP dan LSTM untuk mempelajari pola urutan waktunya.

Dual Classifier Branches: Membagi output menjadi dua cabang, yaitu Gesture Classifier untuk memprediksi jenis gestur tangan dan Domain Classifier untuk mengenali lingkungan/domain.

Uji Forward Pass: Menguji model dengan data tiruan (dummy input) untuk memastikan shape output klasifikasi gestur ([2, 6]) dan domain ([2, 3]) sudah sesuai.

In [15]:
import glob

# Cari semua file .mat di folder /content/ dan seluruh sub-foldernya
all_files = glob.glob('/content/**/*.mat', recursive=True)

print(f"Total file ditemukan: {len(all_files)}")

# Jika file masih sedikit (misal cuma 1-3 file), gunakan semua file tanpa split agar tidak error
if len(all_files) <= 3:
    train_files = all_files
    val_files = all_files
else:
    train_size = int(0.8 * len(all_files))
    val_size = len(all_files) - train_size
    train_files, val_files = torch.utils.data.random_split(all_files, [train_size, val_size])

# Buat DataLoader
train_dataset = WidarBVPDataset(train_files)
val_dataset = WidarBVPDataset(val_files)

# Gunakan batch_size=1 jika sampel file sangat sedikit
batch_sz = 1 if len(all_files) < 4 else 4

train_loader = DataLoader(train_dataset, batch_size=batch_sz, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_sz, shuffle=False)

print(f"Data Train: {len(train_dataset)} sampel | Data Val: {len(val_dataset)} sampel")

Total file ditemukan: 1
Data Train: 1 sampel | Data Val: 1 sampel


Fungsi DataLoader DANN

Membaca & Membagi Dataset: Mengumpulkan file .mat dari direktori lalu membaginya menjadi data latih (80%) dan data validasi (20%).   

Ekstraksi Label Ganda: Memuat matriks BVP sekaligus mengekstrak dua label sekaligus dari nama file, yaitu label gestur tangan ($Y$) dan label lingkungan/domain ($D$).

Batching & Shuffling: Mengelompokkan data ke dalam batch kecil (ukuran 4) dan mengacak urutannya pada data latih agar proses pelatihan model stabil.

In [16]:
loss_gesture_fn = nn.CrossEntropyLoss()
loss_domain_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(dann_model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    dann_model.train()
    running_loss = 0.0
    correct_g = 0
    total = 0

    # Hitung bobot alpha dinamis untuk Gradient Reversal Layer (GRL)
    p = float(epoch) / epochs
    alpha = 2. / (1. + float(torch.exp(torch.tensor(-10. * p)))) - 1.

    for bvp_batch, y_batch, domain_batch in train_loader:
        bvp_batch = bvp_batch.to(device)
        y_batch = y_batch.to(device)
        domain_batch = domain_batch.to(device)

        optimizer.zero_grad()

        # Forward pass
        out_gesture, out_domain = dann_model(bvp_batch, alpha=alpha)

        # Total loss = Loss Gesture + Loss Domain
        err_g = loss_gesture_fn(out_gesture, y_batch)
        err_d = loss_domain_fn(out_domain, domain_batch)
        total_loss = err_g + err_d

        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()
        _, predicted = out_gesture.max(1)
        total += y_batch.size(0)
        correct_g += predicted.eq(y_batch).sum().item()

    train_acc = 100. * correct_g / total

    # Phase Validation
    dann_model.eval()
    val_correct_g = 0
    val_total = 0

    with torch.no_grad():
        for bvp_batch, y_batch, _ in val_loader:
            bvp_batch, y_batch = bvp_batch.to(device), y_batch.to(device)
            out_gesture, _ = dann_model(bvp_batch, alpha=alpha)

            _, predicted = out_gesture.max(1)
            val_total += y_batch.size(0)
            val_correct_g += predicted.eq(y_batch).sum().item()

    val_acc = 100. * val_correct_g / val_total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Total Loss: {running_loss/len(train_loader):.4f} | "
          f"Train Acc Gesture: {train_acc:.2f}% | Val Acc Gesture: {val_acc:.2f}%")

Epoch [1/10] Total Loss: 3.0405 | Train Acc Gesture: 0.00% | Val Acc Gesture: 0.00%
Epoch [2/10] Total Loss: 2.3269 | Train Acc Gesture: 100.00% | Val Acc Gesture: 100.00%
Epoch [3/10] Total Loss: 1.9777 | Train Acc Gesture: 100.00% | Val Acc Gesture: 100.00%
Epoch [4/10] Total Loss: 1.7603 | Train Acc Gesture: 100.00% | Val Acc Gesture: 100.00%
Epoch [5/10] Total Loss: 1.6374 | Train Acc Gesture: 100.00% | Val Acc Gesture: 100.00%
Epoch [6/10] Total Loss: 1.5621 | Train Acc Gesture: 100.00% | Val Acc Gesture: 100.00%
Epoch [7/10] Total Loss: 1.5160 | Train Acc Gesture: 100.00% | Val Acc Gesture: 100.00%
Epoch [8/10] Total Loss: 1.4534 | Train Acc Gesture: 100.00% | Val Acc Gesture: 100.00%
Epoch [9/10] Total Loss: 1.4074 | Train Acc Gesture: 100.00% | Val Acc Gesture: 100.00%
Epoch [10/10] Total Loss: 1.3804 | Train Acc Gesture: 100.00% | Val Acc Gesture: 100.00%


Fungsi Training Loop DANN

Penghitungan Loss Ganda: Menghitung dua jenis error sekaligus, yaitu Gesture Loss (akurasi tebakan gestur) dan Domain Loss (keberhasilan mengenali lingkungan).

Eksekusi Gradient Reversal Layer (GRL): Mengatur nilai $\alpha$ secara dinamis per epoch untuk membalikkan gradien Domain Loss. Hal ini memaksa Feature Extractor menghasilkan fitur yang domain-invariant (bebas dari pengaruh perubahan ruangan/lingkungan).

Pembaruan Bobot (Optimization): Mengoptimalkan bobot jaringan menggunakan Adam berdasarkan total gabungan kedua loss.